In [ ]:
%load_ext autoreload
%autoreload 2

# Imports, setup, data loading

## Imports

In [ ]:
# Imports required packages and functions
import os
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from make_clinical_dataset.epr.combine import merge_closest_measurements
from make_clinical_dataset.shared.constants import ROOT_DIR
from ml_common.summary import get_label_distribution
from ml_common.autogluon import train_models, evaluate

## Setup

In [ ]:
# Show more columns in DataFrame output
pd.set_option('display.max_columns', 100)
pd.set_option("display.max_rows", None)

## Read datasets

In [ ]:
# Path to dated data directory
DATE = '2025-03-29'
DATA_DIR = f"{ROOT_DIR}/data/final/data_{DATE}"
INFO_DIR = f"/cluster/projects/gliugroup/2BLAST/data/info"

TODO: use the latest version of the OACC data

In [ ]:
# Load EHR data
main = pd.read_parquet(f'{DATA_DIR}/processed/treatment_centered_data.parquet')
dates = pd.read_parquet(f'{DATA_DIR}/processed/treatment_centered_dates.parquet')

# Load OACC data
carg = pd.read_csv('/cluster/projects/grantgroup/CTCAE/filtered_df_with_carg.csv', parse_dates=['date_referred'])
carg = carg[['mrn', 'date_referred', 'carg_toxicity_risk']]

# Load CT data
ct = pd.read_parquet("/cluster/home/t128190uhn/datasets/clinical_trials/cleaning/ct.parquet")

# Load lookup tables
religion_names_normalized = pd.read_csv(f'{INFO_DIR}/religion_names_normalized.csv')
language_names_normalized = pd.read_csv(f'{INFO_DIR}/language_names_normalized.csv')

In [ ]:
# Record row and unique MRN counts for patient flow reporting
n_rows_all = len(main)
n_mrns_all = main['mrn'].nunique()

# Define functions locally
Temporary for testing; functions will be moved to the appropriate repo (e.g., ml-commons) once finalized.

In [ ]:
def create_composite_target(
    df,
    component_cols,
    output_col,
    drop_components=True,
):
    """
    Create a composite target with priority:
    1 (any positive) > 0 (any zero) > -1 (all missing)

    Optionally drops component columns after creating the composite.
    """
    any_pos = (df[component_cols] == 1).any(axis=1)
    any_zero = (df[component_cols] == 0).any(axis=1)
    all_missing = (df[component_cols] == -1).all(axis=1)

    df[output_col] = np.select(
        [any_pos, any_zero, all_missing],
        [1, 0, -1],
        default=-1
    )

    # Drop component columns if requested
    if drop_components:
        df.drop(columns=component_cols, inplace=True, errors="ignore")

In [ ]:
def summarize_baseline_characteristics(df):
    """
    Calculate baseline characteristics for older adults:
    - Age buckets
    - Sex
    - Religion
    - Mean height, weight, BSA
    - Cancer type
    - Intent
    """

    out = []

    # ------------------
    # Age buckets
    # ------------------
    age_bins = [65, 70, 75, 80, 85, 90, 95, float("inf")]
    age_labels = [
        "65–69", "70–74", "75–79", "80–84",
        "85–89", "90–94", "95+"
    ]

    age_grp = pd.cut(df["age"], bins=age_bins, labels=age_labels, right=False)
    age_counts = age_grp.value_counts().sort_index()
    age_perc = age_counts / len(df) * 100

    for k in age_labels:
        out.append({
            "Characteristic": "Age group",
            "Category": k,
            "Value": f"{age_counts.get(k, 0)} ({age_perc.get(k, 0):.1f}%)"
        })

    # ------------------
    # Sex
    # ------------------
    sex_counts = df["sex"].value_counts(dropna=False)
    sex_perc = sex_counts / len(df) * 100

    for k, v in sex_counts.items():
        out.append({
            "Characteristic": "Sex",
            "Category": str(k),
            "Value": f"{v} ({sex_perc[k]:.1f}%)"
        })

    # ------------------
    # Religion
    # ------------------
    rel_counts = df["religion"].value_counts(dropna=False)
    rel_perc = rel_counts / len(df) * 100

    for k, v in rel_counts.items():
        out.append({
            "Characteristic": "Religion",
            "Category": str(k),
            "Value": f"{v} ({rel_perc[k]:.1f}%)"
        })

    # ------------------
    # Anthropometrics (mean ± SD)
    # ------------------
    for col, label in [
        ("height", "Height"),
        ("weight", "Weight"),
        ("body_surface_area", "Body surface area"),
    ]:
        out.append({
            "Characteristic": label,
            "Category": "Mean (SD)",
            "Value": f"{df[col].mean():.1f} ({df[col].std():.1f})"
        })

    # ------------------
    # Cancer type
    # ------------------
    ct_counts = df["cancer_type"].value_counts()
    ct_perc = ct_counts / len(df) * 100

    for k, v in ct_counts.items():
        out.append({
            "Characteristic": "Cancer type",
            "Category": k,
            "Value": f"{v} ({ct_perc[k]:.1f}%)"
        })

    # ------------------
    # Intent
    # ------------------
    intent_counts = df["intent"].value_counts()
    intent_perc = intent_counts / len(df) * 100

    for k, v in intent_counts.items():
        out.append({
            "Characteristic": "Intent",
            "Category": k,
            "Value": f"{v} ({intent_perc[k]:.1f}%)"
        })

    return pd.DataFrame(out)


# Target Processing

## Remove unwanted targets

In [ ]:
# Suffixes of target columns to remove
suffixes_to_remove = ("_grade2plus", "_min", "_max", "_30d", "_60d", "_note")

# Collect target columns matching those suffixes and add specific targets to drop explicitly
targets_to_drop = (
    [c for c in main.columns if c.endswith(suffixes_to_remove)]
    + ["target_ED_CTAS_score", "target_H_length_of_stay", "target_ED2H"]
)

# Drop selected columns from the main df in place
main.drop(columns=targets_to_drop, inplace=True, errors="ignore")

## Create composite targets

In [ ]:
# Create a dictionary for composite targets
composite_targets = {
    "esas": {
        "component_cols": [
            c for c in main.columns
            if c.startswith("target_") and c.endswith("_3pt_change")
        ],
        "output_col": "target_any_esas_3pt_deterioration",
    },
    "hematological": {
        "component_cols": [
            "target_hemoglobin_grade3plus",
            "target_neutrophil_grade3plus",
            "target_platelet_grade3plus",
        ],
        "output_col": "target_any_hematological_grade3plus",
    },
    "hepatic": {
        "component_cols": [
            "target_bilirubin_grade3plus",
            "target_ALT_grade3plus",
            "target_AST_grade3plus",
        ],
        "output_col": "target_any_hepatic_grade3plus",
    },
}

In [ ]:
# Apply create_composite_target
for spec in composite_targets.values():
    create_composite_target(
        df=main,
        component_cols=spec["component_cols"],
        output_col=spec["output_col"],
        drop_components=True
    )

TODO: Check lookahead window for ESAS deterioration

TODO: Add creatinine rise, which is a marker of AKI

## Add new targets

TODO: Add targets from CT data

In [ ]:
# Add study-level start date (earliest AE date per study)
ct['study_start_date'] = (
    ct.groupby('study_name')['ae_grade_start_date']
      .transform('min')
)

# Add study-level end date (latest AE date per study)
ct['study_end_date'] = (
    ct.groupby('study_name')['ae_grade_start_date']
      .transform('max')
)

In [ ]:
# ct[['study_name', 'study_start_date', 'study_end_date']].drop_duplicates()

In [ ]:
# ct['study_length_days'] = (
#     ct['study_end_date'] - ct['study_start_date']
# ).dt.days


In [ ]:
# import matplotlib.pyplot as plt
# study_lengths = (
#     ct[['study_name', 'study_length_days']]
#     .drop_duplicates()
# )

# plt.figure()
# plt.hist(study_lengths['study_length_days'].dropna(), bins=30)
# plt.xlabel('Study length (days)')
# plt.ylabel('Number of studies')
# plt.title('Distribution of Study Length (per study)')
# plt.show()


# Preprocessing

## Retrive date columns

In [ ]:
# Add first treatment date from dates table
# Indices are the same
main["first_treatment_date"] = dates["first_treatment_date"]

In [ ]:
# Convert date columns to datetime and remove time component
date_cols = ["assessment_date", "first_treatment_date"]
main[date_cols] = main[date_cols].apply(
    lambda s: pd.to_datetime(s, errors="coerce").dt.floor("D")
)

## Value mapping

In [ ]:
# 1) preferred_language: replace with mapped_language when available
lang_map = (language_names_normalized
            .drop_duplicates("raw_language")
            .set_index("raw_language")["mapped_language"])

main["preferred_language"] = main["preferred_language"].map(lang_map).fillna(main["preferred_language"])

# 2) religion: replace with mapped_lev1_religion_name when available
rel_map = (religion_names_normalized
           .drop_duplicates("raw_religion")
           .set_index("raw_religion")["mapped_lev1_religion_name"])

main["religion"] = main["religion"].map(rel_map).fillna(main["religion"])

In [ ]:
# Collapse rare language categories
s = main["preferred_language"]

# Ccategories with >=250 occurrences
keep = s.value_counts(dropna=False)
keep = keep[keep >= 250].index

# Replace rare categories with "Others" (keep missing as missing)
main["preferred_language"] = s.where(s.isna() | s.isin(keep), "Other")

## Column Grouping

In [ ]:
meta_cols = [
    'mrn', 'assessment_date', 'first_treatment_date', 'primary_site_desc', 'study_drug', 'postalcode'
]
targ_cols = [col for col in main.columns if col.startswith('target') and col not in meta_cols]
feat_cols = main.columns.drop(meta_cols+targ_cols).tolist()

## Filter cohort

### Filter age

In [ ]:
# Filter to patients aged 65+
main_65 = main[main["age"] >= 65].copy()

# Record counts after age filter
n_rows_65 = len(main_65)
n_mrns_65 = main_65["mrn"].nunique()

### Filter first treatment(s)

In [ ]:
# NOTE: This would retain only the very first treatment per patient.
# For now, we keep all first_treatment_date records, so this is commented out.

# get first treatments only
# first_trt_idxs = dates.reset_index().groupby(['mrn','first_treatment_date']).first()['index'].tolist()
# main = main.loc[first_trt_idxs]

In [ ]:
# MRNs with at least one non-null first treatment date (before filtering)
mrns_with_any_first_treatment_date = set(
    main_65.loc[main_65["first_treatment_date"].notna(), "mrn"]
)

# Filter: only rows where assessment_date equals first_treatment_date
main_first_trts = main_65[
    main_65["first_treatment_date"].notna() &
    main_65["assessment_date"].notna() &
    (main_65["first_treatment_date"] == main_65["assessment_date"])
].copy()

# MRNs remaining after date-equality filter
mrns_with_equal_assessment_and_first_treatment_date = set(
    main_first_trts["mrn"]
)

# Counts AFTER the filter (patient-flow reporting)
n_rows_first_trts = len(main_first_trts)
n_mrns_first_trts = len(
    mrns_with_equal_assessment_and_first_treatment_date
)

# MRNs removed by the filter
mrns_removed_by_date_equality_filter = (
    mrns_with_any_first_treatment_date
    - mrns_with_equal_assessment_and_first_treatment_date
)

In [ ]:
# Prepare a summary
summary = {
    "n_mrns_with_any_first_treatment_date": len(mrns_with_any_first_treatment_date),
    "n_mrns_with_equal_dates": len(mrns_with_equal_assessment_and_first_treatment_date),
    "n_mrns_removed_by_date_equality_filter": len(mrns_removed_by_date_equality_filter),
    "percent_removed": (
        len(mrns_removed_by_date_equality_filter)
        / len(mrns_with_any_first_treatment_date)
    ) * 100,
}

summary

TODO: Some mrns_removed_by_date_equality_filter have a cycle 1 record, but the assessment_date doesn’t match the first_treatment_date. I think we should keep these cases, since the presence of cycle 1 should still represent the start of the regimen. For those MRNs, we may need to update first_treatment_date to the date of the cycle 1 start (i.e., the earliest treatment date in cycle 1), because I assume the treatment actually began when the first cycle started, not on the assessment date.

## Standardize regimens (later)

**TODO**

- Standardize regimen records by deriving key schedule fields that are not consistently recorded.

---

**Problem**

- Some regimens include schedule/timing details (e.g., cycle length or day pattern).
- Other regimens lack this information.

---

**Proposed representation of a regimen schedule**

Each regimen schedule is represented using **two components**:

- **Within-cycle day pattern**
  - Set of unique treatment days relative to cycle start
  - Example: `{D1, D2, D10}`

- **Between-cycle length**
  - Computed from gaps between cycle start dates
  - Example: `~14 / 21 / 28 days`

---

**Expected schedule derivation**

Within each stratum:

- `primary_site_desc × morphology_desc × intent × line_of_therapy × regimen`

Steps:
- Identify the most common **within-cycle day pattern(s)**
- Identify the most common **cycle length**
- Use these as the default **expected schedule** when metadata is missing or inconsistent

---

**Open questions**

- **(a)** Should the strata include additional key variables?
- **(b)** What tolerance should be allowed (e.g., ±1–2 days) so minor rescheduling does not create fragmented patterns?

---

**Reference gap**

- Ideally, a reference list of **guideline-defined regimens** with recommended schedules would exist.
- Observed patterns could then be matched to the closest guideline-defined regimen.
- No single structured, comprehensive dataset has been identified so far.

---

**Example**

Assume two guideline-defined regimens for a given:

- `primary_site_desc × morphology_desc × intent × line_of_therapy`

---

**Regimen A — GI-XYZ-TRI (imaginary)**

- Drug: **XYZ**
- Cycle length: **every 30 days**
- Within-cycle pattern: **D1, D3, D10**
- Visits per cycle: **3**

---

**Regimen B — GI-XYZ-WKLY×3 (imaginary)**

- Drug: **XYZ**
- Cycle length: **every 30 days**
- Within-cycle pattern: **D1, D8, D15**
- Schedule: **weekly ×3, then 1 week off**

---

**Patient example (imaginary)**

**Patient 1**

- Primary site: **Colon (GI)**
- Intent: **Palliative**

---

**Treatment dates**

- Cycle 1: **Jan 1, Jan 3, Jan 10**
- Cycle 2: **Feb 1, Feb 3, Feb 10**
- Cycle 3: **Mar 1, Mar 3, Mar 10**

---

**Extracted schedule features**

- **Within-cycle day pattern:** `D1, D3, D10`
- **Between-cycle length:** `~30 days` (allowing small variation)

---

**Result**

- ✅ **Matched regimen:** **GI-XYZ-TRI**
- Reason:
  - Within-cycle pattern matches **D1, D3, D10**
  - Cycle length aligns with **q30**


## Treatment number (later)

**Additional feature proposal**

- Add a feature indicating whether a treatment course is a patient’s **first-ever systemic treatment** or whether they have received **prior treatment(s)**.

---

**Motivation**

- In the current data, ~**51% of MRNs** have more than one distinct treatment start date.
- This suggests many patients underwent **multiple treatment courses**:
  - the same regimen at different times, or
  - multiple different regimens.

---

**Proposed implementation**

- Retain the **start date for each treatment course**.
- Add a binary indicator:
  - `prior_treatment = 1` → patient had at least one systemic treatment **before** this start date
  - `prior_treatment = 0` → no prior systemic treatment before this start date

---

**Question**

- Would you agree with this approach for capturing prior treatment exposure?

---


# Data Integration

## Join EHR with OACC

In [ ]:
# Closest referral within 90 days BEFORE first treatment (per mrn x first_treatment_date row)
main_first_trts = merge_closest_measurements(
    main_first_trts,
    carg,
    main_date_col="first_treatment_date",
    meas_date_col="date_referred",
    direction="backward",
    time_window=(-90, 0),
    merge_individually=False,
    include_meas_date=True,   # keeps date_referred in output
)

In [ ]:
# Add more columns to meta_cols
meta_cols += ['date_referred', 'carg_toxicity_risk']

## Correct CARG Referral Matching Logic

TODO: Drop CARG scores from later treatments after earliest match

---

**Example: Multiple treatment courses for a single MRN**

Assume **MRN = 12345** has **4 distinct treatment courses**, each with a different `treatment_start_date`.

---

**Treatment history**

| Treatment course | Treatment start date | Days since previous | CARG referral date | Matched by rule |
|------------------|---------------------|---------------------|--------------------|-----------------|
| Regimen 1        | Jan 1               | –                   | Feb 15             | ✅ (within 90 days) |
| Regimen 2        | Feb 20              | +50 days            | Feb 15             | ✅ (within 90 days) |
| Regimen 3        | May 10              | +79 days            | Feb 15             | ❌ (outside 90 days) |
| Regimen 4        | Aug 1               | +83 days            | Feb 15             | ❌ (outside 90 days) |

---

**Matching rule**

- A CARG referral is matched to a treatment course if it occurs **within 90 days before** the treatment start date.

---

**Observed behavior**

- The same CARG referral (**Feb 15**) falls within 90 days of:
  - **Regimen 1** (Jan 1)
  - **Regimen 2** (Feb 20)
- As a result, the referral is **matched to both treatment courses** under the current rule.

---

**Clinical interpretation**

- The CARG referral is clearly intended for the **first-ever systemic treatment**.
- The match to **Regimen 2** is a byproduct of temporal proximity, not clinical intent.

---

**Implication**

- After time-window matching:
  - Retain the CARG score **only for the earliest matched treatment per MRN**
  - Remove the CARG score from **subsequent treatments** for the same MRN

# Data Splitting

In [ ]:
# Set split date using the earliest assessment date with non-missing CARG
split_date = main_first_trts['date_referred'].min()
print(f'Temporal split at {split_date}')

In [ ]:
# Define mask for test set based on first treatment date
mask = main_first_trts["first_treatment_date"] >= split_date

# Apply split using mask
dev = main_first_trts[~mask].copy()
test = main_first_trts[mask].copy()

In [ ]:
# NOTE: Currently disabled.
# We keep all first_treatment_date records; enabling this would remove 879 sessions.

# (Prevents MRN leakage by excluding test MRNs from dev.)
# dev = dev[~dev["mrn"].isin(test["mrn"])]

In [ ]:
# Ensure no columns are left unclassified (meta / feature / target)
extra_cols = (
    set(main_first_trts.columns)
    - set(meta_cols)
    - set(feat_cols)
    - set(targ_cols)
)

assert not extra_cols, f"Unassigned columns found: {sorted(extra_cols)}"

In [ ]:
# Split dev data into metadata, targets, and features
dev_meta, dev_target, dev_feats = dev[meta_cols].copy(), dev[targ_cols].copy(), dev[feat_cols].copy()

# Split test data into metadata, targets, and features
test_meta, test_target, test_feats = test[meta_cols].copy(), test[targ_cols].copy(), test[feat_cols].copy()

# Label dataset split
dev_meta['split'] = 'Dev'
test_meta['split'] = 'Test'

# Combine dev and test for downstream processing
X, Y, meta = pd.concat([dev_feats, test_feats]), pd.concat([dev_target, test_target]), pd.concat([dev_meta, test_meta])

# Patient Flow

In [ ]:
# Dev counts
n_rows_dev = len(dev)
n_mrns_dev = dev["mrn"].nunique()

# Test counts
n_rows_test = len(test)
n_mrns_test = test["mrn"].nunique()

# Test rows/MRNs with available CARG (no subsetting created)
carg_mask = test["carg_toxicity_risk"].notna()

n_rows_test_carg = carg_mask.sum()
n_mrns_test_carg = test.loc[carg_mask, "mrn"].nunique()

In [ ]:
patient_flow_table = pd.DataFrame(
    [
        ("All data", n_rows_all, n_mrns_all),
        ("Age ≥ 65", n_rows_65, n_mrns_65),
        ("First treatment only", n_rows_first_trts, n_mrns_first_trts),
        ("Development set", n_rows_dev, n_mrns_dev),
        ("Test set", n_rows_test, n_mrns_test),
        ("Test set with CARG score", n_rows_test_carg, n_mrns_test_carg),
    ],
    columns=["Step", "Rows", "Unique MRNs"]
)

patient_flow_table

The discrepancy between the number of rows and unique MRNs in the test set with available CARG scores is due to a known issue in the matching logic, which will be addressed in the “Correct CARG Referral Matching Logic” section soon.

TODO: Confirm whether having substantially more records in dev than in test is okay.

TODO: Find MRNs with no chemo info

# Label distribution

In [ ]:
# Per session
get_label_distribution(Y, meta, with_respect_to='sessions')

In [ ]:
# Per patient
get_label_distribution(Y, meta, with_respect_to='patients')

In [ ]:
root_dir = '/cluster/projects/gliugroup/work_dir/elnaz_ziad'

# Baseline Charachteristics

TODO: Define a function to report characteristics for a selected dataset (overall, development, or test).

In [ ]:
baseline_table = summarize_baseline_characteristics(main_first_trts)
baseline_table

TODO: Break the Christian category into subcategories, as it is the dominant religion and may not be informative in its current aggregated form.

<div style="color: red; font-size: 22px; font-weight: bold;">
⚠️ The rest of the notebook is still being organized and may change, so please stop here for now. Thank you. :D
</div>

In [ ]:
# main_first_trts.columns.to_list()

# Exploratory Data Analysis (EDA)

# Model Development

# Model Evaluation

# Comparative Analysis

# GO-TREAT - v2

In [ ]:
models = train_models(
    dev_feats, 
    dev_target.drop(columns=['target_ED_30d', 'target_ED_60d', 'target_ED_90d']), # already computed
    dev_meta, 
    save_path=f'{root_dir}/AutogluonModels/go-treat-v2/'
)

In [ ]:
models = {}
for target in Y.columns:
    model_dir = f'{root_dir}/AutogluonModels/go-treat-v2/{target}-medium-average_precision'
    if os.path.exists(model_dir):
        models[target] = TabularPredictor.load(model_dir, verbosity=0)

In [ ]:
res = {}
for target in models:
    res[target] = models[target].leaderboard()[['model', 'score_val']]
pd.concat(res, axis=1)

In [ ]:
evaluate(models, test_feats, test_target)

In [ ]:
# AUC of CARG
from ml_common.eval import auc_scores
mask = test_meta['carg_toxicity_risk'].notna()
carg_score = test_meta.loc[mask, 'carg_toxicity_risk'].replace({'Low': '1', 'Moderate': '2', 'High': '3'}).astype(int)
res = {}
for target, label in test_target[mask].items():
    res[target] = auc_scores(label[label != -1], carg_score[label != -1])
pd.DataFrame(res)

In [ ]:
print("Test Data with CARG Score (after anchoring CARG to treatments with a lookback window of 90 days)")
print(f"Pateints = {test_meta.loc[mask, 'mrn'].nunique()}")
print(f"Sessions = {len(test_meta[mask])}")

# GO-TREAT - v1

In [ ]:
print("A2R Data")
print(f"Pateints = {main['mrn'].nunique()}")
print(f"Sessions = {len(main)}")

print("\nCarg Data")
print(f"Pateints = {carg['mrn'].nunique()}")
print(f"Sessions = {len(carg)}")

In [ ]:
print("A2R Data")
print(f"Pateints (Baseline Age >= 65) = {sum(main.groupby('mrn').first()['age'] >= 65)}")

In [ ]:
main.groupby('mrn').first()['assessment_date'].dt.year.value_counts().sort_index().plot(kind='bar')

In [ ]:
carg.groupby('mrn').first()['date_referred'].dt.year.value_counts().sort_index().plot(kind='bar')

In [ ]:
# get the develpment and test set
main = merge_closest_measurements(
    main, carg, main_date_col='assessment_date', meas_date_col='date_referred', time_window=(-90,0), 
    merge_individually=False
)
mask = main['date_referred'].notna()
dev, test = main[~mask].copy(), main[mask].copy()
dev = dev[~dev['mrn'].isin(test['mrn'])] # don't include mrns from test set
meta_cols += ['date_referred', 'carg_toxicity_risk']

In [ ]:
dev_meta, dev_target, dev_feats = dev[meta_cols].copy(), dev[targ_cols].copy(), dev[feat_cols].copy()
test_meta, test_target, test_feats = test[meta_cols].copy(), test[targ_cols].copy(), test[feat_cols].copy()
dev_meta['split'] = 'Dev'
test_meta['split'] = 'Test'

X, Y, meta = pd.concat([dev_feats, test_feats]), pd.concat([dev_target, test_target]), pd.concat([dev_meta, test_meta])

In [ ]:
print("Test Data (after anchoring CARG to treatments with a lookback window of 90 days)")
print(f"Pateints = {test_meta['mrn'].nunique()}")
print(f"Sessions = {len(test_meta)}")

In [ ]:
get_label_distribution(Y, meta, with_respect_to='sessions')

In [ ]:
get_label_distribution(Y, meta, with_respect_to='patients')

In [ ]:
# !python ~/slurm/go-treat.py

In [ ]:
import os
root_dir = '/cluster/projects/gliugroup/work_dir/kevin_he'
models = {}
for target in Y.columns:
    model_dir = f'{root_dir}/AutogluonModels/go-treat-v1/{target}-medium-average_precision'
    if os.path.exists(model_dir):
        models[target] = TabularPredictor.load(model_dir, verbosity=0)

In [ ]:
res = {}
for target in models:
    res[target] = models[target].leaderboard()[['model', 'score_val']]
pd.concat(res, axis=1)

In [ ]:
res = {}
for target in models:
    res[target] = models[target].leaderboard()[['model', 'score_val']]
pd.concat(res, axis=1)

### Baseline Evaluation

In [ ]:
def evaluate(
    models: dict[str, TabularPredictor],
    X: pd.DataFrame,
    Y: pd.DataFrame,
    return_full: bool = False,
) -> pd.DataFrame:
    """Evaluate performance for all targets and all model types

    Args:
        return_full: If True, return the full information about models (training times, inference times, stack levels, etc)
    """
    results = {}
    for target, model in models.items():
        mask = Y[target] != -1
        if Y.loc[mask, target].nunique() == 1: continue
        
        data = pd.concat([X[mask], Y.loc[mask, target]], axis=1)
        res = model.leaderboard(data, extra_metrics=["roc_auc", "average_precision"])
        results[target] = (
            res if return_full else res[["model", "roc_auc", "average_precision"]]
        )
    results = pd.concat(results, axis=1)
    return results

In [ ]:
baseline_idxs = test_meta.reset_index().groupby('mrn').first()['index'].tolist()

In [ ]:
evaluate(models, test_feats.loc[baseline_idxs], test_target.loc[baseline_idxs])

In [ ]:
# AUC of CARG
from ml_common.eval import auc_scores
carg_score = test_meta.loc[baseline_idxs, 'carg_toxicity_risk'].replace({'Low': '1', 'Moderate': '2', 'High': '3'}).astype(int)
res = {}
for target, label in test_target.loc[baseline_idxs].items():
    mask = label != -1
    res[target] = auc_scores(label[mask], carg_score[mask])
pd.DataFrame(res)